# APIM ❤️ AI Foundry

## Multi-Model Failover lab

This lab demonstrates automatic failover between different AI models using Azure API Management with priority-based routing, retry policies with exponential backoff, circuit breaker patterns, built-in LLM logging, FinOps cost controls, and Microsoft Agent Framework (MAF) agent testing.

### Key Features
- **Backend pool** with priority-based routing across gpt-4.1-nano (primary), gpt-5.2 (secondary), and gpt-4.1 (tertiary)
- **Retry policy** with exponential backoff for 429/503 errors
- **Circuit breaker** to temporarily remove unhealthy backends
- **Built-in LLM logging** to track usage across all backends
- **FinOps framework** with per-product token rate limiting and cost quotas
- **MAF agent testing** through the APIM gateway
- **Three APIM products** (Finance, Marketing, HR) with dedicated subscriptions

### Prerequisites

- [Python 3.12 or later version](https://www.python.org/) installed
- [VS Code](https://code.visualstudio.com/) installed with the [Jupyter notebook extension](https://marketplace.visualstudio.com/items?itemName=ms-toolsai.jupyter) enabled
- [Python environment](https://code.visualstudio.com/docs/python/environments#_creating-environments) with the [requirements.txt](../../../requirements.txt) or run `pip install -r requirements.txt` in your terminal
- [An Azure Subscription](https://azure.microsoft.com/free/) with [Contributor](https://learn.microsoft.com/en-us/azure/role-based-access-control/built-in-roles/privileged#contributor) + [RBAC Administrator](https://learn.microsoft.com/en-us/azure/role-based-access-control/built-in-roles/privileged#role-based-access-control-administrator) or [Owner](https://learn.microsoft.com/en-us/azure/role-based-access-control/built-in-roles/privileged#owner) roles
- [Azure CLI](https://learn.microsoft.com/cli/azure/install-azure-cli) installed and [Signed into your Azure subscription](https://learn.microsoft.com/cli/azure/authenticate-azure-cli-interactively)

▶️ Click `Run All` to execute all steps sequentially, or execute them `Step by Step`...

<a id='0'></a>
### 0️⃣ Initialize notebook variables

- Resources will be suffixed by a unique string based on your subscription id.
- Adjust the location parameters according your preferences and on the [product availability by Azure region.](https://azure.microsoft.com/explore/global-infrastructure/products-by-region/?cdn=disable&products=cognitive-services,api-management) 
- Adjust the OpenAI model and version according the [availability by region.](https://learn.microsoft.com/azure/ai-services/openai/concepts/models)

In [1]:
import os, sys, json
sys.path.insert(1, '../../shared')  # add the shared directory to the Python path
import utils

deployment_name = os.path.basename(os.path.dirname(globals()['__vsc_ipynb_file__']))
resource_group_name = f"lab-{deployment_name}"
resource_group_location = "westeurope"

# AI Services - two regions for failover diversity
aiservices_config = [{"name": "foundry1", "location": "swedencentral", "priority": 1},
                     {"name": "foundry2", "location": "eastus2", "priority": 2}]

# Models - three models with priority-based failover
# gpt-4.1-nano (primary) -> gpt-5.2 (secondary) -> gpt-4.1 (tertiary)
models_config = [
    {"name": "gpt-4.1-nano", "publisher": "OpenAI", "version": "2025-04-14", "sku": "GlobalStandard", "capacity": 200,
     "inputTokensMeterSku": "gpt 4.1 nano Inp glbl", "outputTokensMeterSku": "gpt 4.1 nano Outp glbl"},
    {"name": "gpt-5.2", "publisher": "OpenAI", "version": "2025-12-11", "sku": "GlobalStandard", "capacity": 200,
     "inputTokensMeterSku": "gpt 5 pro inp glbl", "outputTokensMeterSku": "gpt 5 pro out glbl"},
    {"name": "gpt-4.1", "publisher": "OpenAI", "version": "2025-04-14", "sku": "GlobalStandard", "capacity": 200,
     "inputTokensMeterSku": "gpt 4.1 Inp glbl", "outputTokensMeterSku": "gpt 4.1 Outp glbl"}
]

apim_sku = 'Basicv2'

# Products with per-product token rate limits and cost quotas
apim_products_config = [
    {"name": "finance", "displayName": "Finance Product", "tpm": 2000, "tokenQuota": 1500000, "tokenQuotaPeriod": "Monthly", "costQuota": 20},
    {"name": "marketing", "displayName": "Marketing Product", "tpm": 1000, "tokenQuota": 1000000, "tokenQuotaPeriod": "Monthly", "costQuota": 10},
    {"name": "hr", "displayName": "HR Product", "tpm": 500, "tokenQuota": 500000, "tokenQuotaPeriod": "Monthly", "costQuota": 5}
]

# Product-scoped subscriptions
apim_subscriptions_config = [
    {"name": "subscription-finance", "displayName": "Finance Subscription", "product": "finance"},
    {"name": "subscription-marketing", "displayName": "Marketing Subscription", "product": "marketing"},
    {"name": "subscription-hr", "displayName": "HR Subscription", "product": "hr"}
]

inference_api_path = "inference"
inference_api_type = "AzureOpenAI"
inference_api_version = "2025-03-01-preview"
foundry_project_name = deployment_name

currency_code = 'USD'

utils.print_ok('Notebook initialized')

✅ Notebook initialized ⌚ 17:35:34.077605 


In [3]:
print(f"Deployment Name: {deployment_name}")

Deployment Name: multi-model-failover


<a id='1'></a>
### 1️⃣ Verify the Azure CLI and the connected Azure subscription

The following commands ensure that you have the latest version of the Azure CLI and that the Azure CLI is connected to your Azure subscription.

In [2]:
output = utils.run("az account show", "Retrieved az account", "Failed to get the current az account")

if output.success and output.json_data:
    current_user = output.json_data['user']['name']
    tenant_id = output.json_data['tenantId']
    subscription_id = output.json_data['id']

    utils.print_info(f"Current user: {current_user}")
    utils.print_info(f"Tenant ID: {tenant_id}")
    utils.print_info(f"Subscription ID: {subscription_id}")

⚙️ Running: az account show 
✅ Retrieved az account ⌚ 17:36:35.739020 :31s]
👉🏽 Current user: admin@MngEnvMCAP664615.onmicrosoft.com
👉🏽 Tenant ID: c6f69f43-60c3-42be-aee3-e4985c499e45
👉🏽 Subscription ID: de281c5e-5d60-4fc1-b905-c91caf45e624


<a id='2'></a>
### 2️⃣ Create deployment using 🦾 Bicep

This lab uses [Bicep](https://learn.microsoft.com/azure/azure-resource-manager/bicep/overview?tabs=bicep) to declarative define all the resources that will be deployed in the specified resource group. Change the parameters or the [main.bicep](main.bicep) directly to try different configurations.

The deployment creates:
- **Backend pool** with priority-based routing and circuit breaker
- **Three APIM products** (Finance, Marketing, HR) with per-product token rate limiting
- **Product-scoped subscriptions** for each department
- **FinOps resources**: pricing table, subscription quota table, Azure Monitor workbooks, and automated alerts
- **Built-in LLM logging** for tracking token usage across all backends

⚠️ Retry this step if you get deployment error: `workspace not active`

In [4]:
# Create the resource group if doesn't exist
utils.create_resource_group(resource_group_name, resource_group_location)

# Define the Bicep parameters
bicep_parameters = {
    "": "https://schema.management.azure.com/schemas/2019-04-01/deploymentParameters.json#",
    "contentVersion": "1.0.0.0",
    "parameters": {
        "apimSku": { "value": apim_sku },
        "aiServicesConfig": { "value": aiservices_config },
        "modelsConfig": { "value": models_config },
        "apimSubscriptionsConfig": { "value": apim_subscriptions_config },
        "apimProductsConfig": { "value": apim_products_config },
        "inferenceAPIPath": { "value": inference_api_path },
        "inferenceAPIType": { "value": inference_api_type },
        "foundryProjectName": { "value": foundry_project_name }
    }
}

# Write the parameters to the params.json file
with open('params.json', 'w') as bicep_parameters_file:
    bicep_parameters_file.write(json.dumps(bicep_parameters))

# Run the deployment
output = utils.run(f"az deployment group create --name {deployment_name} --resource-group {resource_group_name} --template-file main.bicep --parameters params.json",
    f"Deployment '{deployment_name}' succeeded", f"Deployment '{deployment_name}' failed")

⚙️ Running: az group show --name lab-multi-model-failover 
👉🏽 Resource group lab-multi-model-failover does not yet exist. Creating the resource group now...
⚙️ Running: az group create --name lab-multi-model-failover --location westeurope --tags source=ai-gateway 
✅ Resource group 'lab-multi-model-failover' created ⌚ 17:38:02.101118 :3s]
⚙️ Running: az deployment group create --name multi-model-failover --resource-group lab-multi-model-failover --template-file main.bicep --parameters params.json 
❌ Deployment 'multi-model-failover' failed ⌚ 17:42:18.869695 :16s] WARNING: A new Bicep release is available: v0.41.2. Upgrade now by running "az bicep upgrade".
/Users/mimarusa/Documents/PRJ/_DEMOS/AI-Gateway/modules/cognitive-services/v3/deployments.bicep(13,26) : Warning BCP081: Resource type "Microsoft.CognitiveServices/accounts/deployments@2025-06-01" does not have types available. Bicep is unable to validate resource properties prior to deployment, but this will not block the resource fr

<a id='3'></a>
### 3️⃣ Get the deployment outputs

Retrieve the gateway URL, subscription keys, and FinOps resource endpoints from the Bicep deployment.

In [ ]:
# Obtain all of the outputs from the deployment
output = utils.run(f"az deployment group show --name {deployment_name} -g {resource_group_name}", f"Retrieved deployment: {deployment_name}", f"Failed to retrieve deployment: {deployment_name}")

if output.success and output.json_data:
    apim_resource_gateway_url = utils.get_deployment_output(output, 'apimResourceGatewayURL', 'APIM API Gateway URL')
    app_insights_name = utils.get_deployment_output(output, 'appInsightsName', 'App Insights Name')
    foundry_project_endpoint = utils.get_deployment_output(output, 'foundryProjectEndpoint', 'Foundry Project Endpoint')
    pricing_dcr_endpoint = utils.get_deployment_output(output, 'pricingDCREndpoint', 'Pricing DCR Endpoint')
    pricing_dcr_immutable_id = utils.get_deployment_output(output, 'pricingDCRImmutableId', 'Pricing DCR ImmutableId')
    pricing_dcr_stream = utils.get_deployment_output(output, 'pricingDCRStream', 'Pricing DCR Stream')
    subscription_quota_dcr_endpoint = utils.get_deployment_output(output, 'subscriptionQuotaDCREndpoint', 'Subscription Quota DCR Endpoint')
    subscription_quota_dcr_immutable_id = utils.get_deployment_output(output, 'subscriptionQuotaDCRImmutableId', 'Subscription Quota DCR ImmutableId')
    subscription_quota_dcr_stream = utils.get_deployment_output(output, 'subscriptionQuotaDCRStream', 'Subscription Quota DCR Stream')

    apim_subscriptions = json.loads(utils.get_deployment_output(output, 'apimSubscriptions').replace("'", "\""))
    for subscription in apim_subscriptions:
        subscription_name = subscription['name']
        subscription_key = subscription['key']
        utils.print_info(f"Subscription Name: {subscription_name}")
        utils.print_info(f"Subscription Key: ****{subscription_key[-4:]}")

In [5]:
# Obtain all of the outputs from the deployment
output = utils.run(f"az deployment group show --name {deployment_name} -g {resource_group_name}", f"Retrieved deployment: {deployment_name}", f"Failed to retrieve deployment: {deployment_name}")

if output.success and output.json_data:
    apim_resource_gateway_url = utils.get_deployment_output(output, 'apimResourceGatewayURL', 'APIM API Gateway URL')
    app_insights_name = utils.get_deployment_output(output, 'appInsightsName', 'App Insights Name')
    foundry_project_endpoint = utils.get_deployment_output(output, 'foundryProjectEndpoint', 'Foundry Project Endpoint')
    pricing_dcr_endpoint = utils.get_deployment_output(output, 'pricingDCREndpoint', 'Pricing DCR Endpoint')
    pricing_dcr_immutable_id = utils.get_deployment_output(output, 'pricingDCRImmutableId', 'Pricing DCR ImmutableId')
    pricing_dcr_stream = utils.get_deployment_output(output, 'pricingDCRStream', 'Pricing DCR Stream')
    subscription_quota_dcr_endpoint = utils.get_deployment_output(output, 'subscriptionQuotaDCREndpoint', 'Subscription Quota DCR Endpoint')
    subscription_quota_dcr_immutable_id = utils.get_deployment_output(output, 'subscriptionQuotaDCRImmutableId', 'Subscription Quota DCR ImmutableId')
    subscription_quota_dcr_stream = utils.get_deployment_output(output, 'subscriptionQuotaDCRStream', 'Subscription Quota DCR Stream')

    apim_subscriptions = json.loads(utils.get_deployment_output(output, 'apimSubscriptions').replace("'", "\""))
    for subscription in apim_subscriptions:
        subscription_name = subscription['name']
        subscription_key = subscription['key']
        utils.print_info(f"Subscription Name: {subscription_name}")
        utils.print_info(f"Subscription Key: ****{subscription_key[-4:]}")

⚙️ Running: az deployment group show --name multi-model-failover -g lab-multi-model-failover 
✅ Retrieved deployment: multi-model-failover ⌚ 21:51:10.369752 :23s]
👉🏽 APIM API Gateway URL: https://apim-g3mjvytixvcoc.azure-api.net
👉🏽 App Insights Name: insights-g3mjvytixvcoc
👉🏽 Foundry Project Endpoint: https://foundry1-g3mjvytixvcoc.services.ai.azure.com/api/projects/multi-model-failover-foundry1
👉🏽 Pricing DCR Endpoint: https://dcr-pricing-g3mjvytixvcoc-tjlt-westeurope.logs.z1.ingest.monitor.azure.com
👉🏽 Pricing DCR ImmutableId: dcr-691bee764c9c4ad0aab7cc213cec2b08
👉🏽 Pricing DCR Stream: Custom-Json-PRICING_CL
👉🏽 Subscription Quota DCR Endpoint: https://dcr-quota-g3mjvytixvcoc-0w7x-westeurope.logs.z1.ingest.monitor.azure.com
👉🏽 Subscription Quota DCR ImmutableId: dcr-3a67c66c875042cd9213c1dca9b755e3
👉🏽 Subscription Quota DCR Stream: Custom-Json-SUBSCRIPTION_QUOTA_CL
👉🏽 Subscription Name: subscription-finance
👉🏽 Subscription Key: ****8660
👉🏽 Subscription Name: subscription-marketing
👉🏽 

In [18]:
apim_resource_gateway_url

'https://apim-g3mjvytixvcoc.azure-api.net'

<a id='4'></a>
### 4️⃣ Load the pricing data into Azure Monitor custom table

👉 This script uses retail price information. Please adjust it to apply a discount or to use a flat rate with PTUs.   
👉 We are multiplying by 1000 to get the retail price per 1K tokens.   
👉 Deploy this script as a [job](https://learn.microsoft.com/en-us/azure/container-apps/jobs?tabs=azure-cli) to run automatically on a predefined schedule.

In [6]:
import requests
from azure.identity import DefaultAzureCredential
from azure.monitor.ingestion import LogsIngestionClient
from azure.core.exceptions import HttpResponseError
from datetime import datetime, timezone

credential = DefaultAzureCredential()
client = LogsIngestionClient(endpoint=pricing_dcr_endpoint, credential=credential, logging_enable=False)

for aiservice in aiservices_config:
    aiservice_resource_location = aiservice['location']
    prices = requests.get(f"https://prices.azure.com/api/retail/prices?currencyCode='{currency_code}'&=serviceName eq 'Foundry Models' and unitOfMeasure eq '1K' and armRegionName eq '{aiservice_resource_location}'")
    if prices.status_code == 200:
        prices_json = prices.json()
        if prices_json and 'Items' in prices_json:
            for deployment in models_config:
                input_tokens_price = next((item['retailPrice'] * 1000 for item in prices_json['Items'] if item.get('skuName') == deployment.get("inputTokensMeterSku")), None)
                output_tokens_price = next((item['retailPrice'] * 1000 for item in prices_json['Items'] if item.get('skuName') == deployment.get("outputTokensMeterSku")), None)
                utils.print_info(f"Adding model {deployment.get('name')} with input / output tokens price {input_tokens_price} / {output_tokens_price}")
                body = [{ "TimeGenerated": str(datetime.now(timezone.utc)),
                        "Model": deployment.get("name"),
                        "InputTokensPrice": input_tokens_price,
                        "OutputTokensPrice": output_tokens_price }]
                try:
                    client.upload(rule_id=pricing_dcr_immutable_id, stream_name=pricing_dcr_stream, logs=body)
                    utils.print_ok(f"Upload succeeded for model {deployment.get('name')}")
                except HttpResponseError as e:
                    utils.print_error(f"Upload failed: {e}")

👉🏽 Adding model gpt-4.1-nano with input / output tokens price None / None
✅ Upload succeeded for model gpt-4.1-nano ⌚ 21:51:45.086261 
👉🏽 Adding model gpt-5.2 with input / output tokens price None / None
✅ Upload succeeded for model gpt-5.2 ⌚ 21:51:45.182375 
👉🏽 Adding model gpt-4.1 with input / output tokens price None / None
✅ Upload succeeded for model gpt-4.1 ⌚ 21:51:45.384459 
👉🏽 Adding model gpt-4.1-nano with input / output tokens price None / None
✅ Upload succeeded for model gpt-4.1-nano ⌚ 21:51:45.996560 
👉🏽 Adding model gpt-5.2 with input / output tokens price None / None
✅ Upload succeeded for model gpt-5.2 ⌚ 21:51:46.112269 
👉🏽 Adding model gpt-4.1 with input / output tokens price None / None
✅ Upload succeeded for model gpt-4.1 ⌚ 21:51:46.332714 


<a id='5'></a>
### 5️⃣ Load the Subscription Quota into Azure Monitor custom table

This uploads the cost quota for each subscription so Azure Monitor alerts can automatically suspend subscriptions that exceed their budget.

In [7]:
from azure.identity import DefaultAzureCredential
from azure.monitor.ingestion import LogsIngestionClient
from azure.core.exceptions import HttpResponseError
from datetime import datetime, timezone

credential = DefaultAzureCredential()
client = LogsIngestionClient(endpoint=subscription_quota_dcr_endpoint, credential=credential, logging_enable=False)

for subscription in apim_subscriptions_config:
    for product in apim_products_config:
        if product.get("name") == subscription.get("product"):
            cost_quota = product.get("costQuota")
            utils.print_info(f"Adding {subscription.get('name')} with cost quota ")
            body = [{
                "TimeGenerated": str(datetime.now(timezone.utc)),
                "Subscription": subscription.get("name"),
                "CostQuota": cost_quota
            }]
            try:
                client.upload(rule_id=subscription_quota_dcr_immutable_id, stream_name=subscription_quota_dcr_stream, logs=body)
                utils.print_ok(f"Upload succeeded for {subscription.get('name')}")
            except HttpResponseError as e:
                utils.print_error(f"Upload failed: {e}")

👉🏽 Adding subscription-finance with cost quota 
✅ Upload succeeded for subscription-finance ⌚ 21:52:25.954212 
👉🏽 Adding subscription-marketing with cost quota 
✅ Upload succeeded for subscription-marketing ⌚ 21:52:26.130084 
👉🏽 Adding subscription-hr with cost quota 
✅ Upload succeeded for subscription-hr ⌚ 21:52:26.285838 


<a id='sdk'></a>
### 🧪 Test failover using the Azure OpenAI Python SDK

Send requests across all subscriptions and models to test priority-based failover. Requests route to gpt-4.1-nano first; if throttled (429) or unavailable (503), the retry policy with exponential backoff triggers failover to gpt-5.2, then gpt-4.1.

👉 Adjust the `sleep_time_ms` and the number of `runs` to your test scenario.

In [14]:
import time, random
from openai import AzureOpenAI

runs = 10
sleep_time_ms = 100

for i in range(runs):
    apim_subscription = random.choice(apim_subscriptions)
    openai_model = random.choice(models_config)
    client = AzureOpenAI(
        azure_endpoint = f"{apim_resource_gateway_url}/{inference_api_path}",
        api_key = apim_subscription.get("key"),
        api_version = inference_api_version
    )
    try:
        response = client.chat.completions.create(
            model = str(openai_model.get('name')),
            messages = [
                {"role": "user", "content": "Can you tell me the time, please?"}
            ],
            extra_headers = {"x-user-id": "alex"}
        )
        print(f"▶️ Run {i+1}/{runs}: [{apim_subscription.get('name')} w/ {openai_model.get('name')}] 💬 {response.choices[0].message.content}")
    except Exception as e:
        print(f"❌ Run {i+1}/{runs}: [{apim_subscription.get('name')} w/ {openai_model.get('name')}] Error: {e}")
    time.sleep(sleep_time_ms/1000)

▶️ Run 1/10: [subscription-marketing w/ gpt-5.2] 💬 I can’t directly see your local time. If you tell me your city/time zone (or share your current UTC offset), I’ll tell you the current time there.
▶️ Run 2/10: [subscription-hr w/ gpt-4.1] 💬 I'm sorry, but I don't have access to the current time. Please check your device's clock or ask a voice assistant like Siri or Google Assistant!
▶️ Run 3/10: [subscription-hr w/ gpt-5.2] 💬 I can’t see your local time from here. If you tell me your city/time zone (e.g., “Berlin” or “UTC+2”), I’ll tell you the current time there.
▶️ Run 4/10: [subscription-finance w/ gpt-5.2] 💬 I can’t see your local clock from here. If you tell me your city/time zone (or share your current UTC offset), I’ll tell you the current time there.
▶️ Run 5/10: [subscription-marketing w/ gpt-4.1-nano] 💬 I'm sorry, but I can't provide the current time. Please check the clock on your device.
▶️ Run 6/10: [subscription-hr w/ gpt-5.2] 💬 I can’t directly see your current local ti

<a id='agent'></a>
### 🤖 Test with Microsoft Agent Framework (MAF) agent

Create and run an AI agent using the [Azure AI Agent Service](https://learn.microsoft.com/en-us/azure/ai-services/agents/overview) through the APIM gateway. The agent uses the same backend pool with priority-based failover, demonstrating that the failover pattern works transparently for agent workloads.

In [ ]:
# from azure.ai.projects import AIProjectClient
# from azure.identity import DefaultAzureCredential
# # from azure.ai.projects.models import PromptAgentDefinition, CodeInterpreterTool, CodeInterpreterToolAuto

# project_client = AIProjectClient(
#     endpoint=foundry_project_endpoint,
#     credential=DefaultAzureCredential(),
# )
# agents_client = project_client.agents

# code_interpreter = CodeInterpreterTool(container=CodeInterpreterToolAuto(file_ids=[]))

# with project_client:
#     agent = agents_client.create_version(
#         definition=PromptAgentDefinition(
#             model=str(models_config[0].get('name')),
#             instructions="You are a helpful financial analyst assistant. Answer questions concisely.",
#             # tools=[code_interpreter]
#         ),
#         agent_name="failover-test-agent"
#     )
#     utils.print_ok(f"Created agent, ID: {agent.id}")

#     openai_client = project_client.get_openai_client()
#     conversation_id = openai_client.conversations.create().id
#     utils.print_ok(f"Created conversation, ID: {conversation_id}")

#     # Test the agent with a financial analysis question
#     response = openai_client.responses.create(
    #     input=[{"role": "user", "content": "Calculate the compound interest on 0,000 at 5% annual rate for 3 years, compounded monthly."}],
    #     conversation=conversation_id,
    #     extra_body={"agent": {"name": agent.name, "type": "agent_reference"}},
    # )

    # print(f"🗨️ Agent: {response.output_text}")

<a id='tokens'></a>
### 📊 Analyze token usage by subscription and agent

Query Application Insights to see token consumption across different subscriptions and agent executions. This data feeds into the FinOps framework for cost tracking.

In [ ]:
import pandas as pd

query = (
    "customMetrics "
    "| where name == 'Total Tokens' "
    "| where timestamp >= ago(4h) "
    "| extend parsedCustomDimensions = parse_json(customDimensions) "
    "| extend apimSubscription = tostring(parsedCustomDimensions.['Subscription ID']) "
    "| extend agentID = tostring(parsedCustomDimensions.['Agent ID']) "
    "| summarize TotalValue = sum(value) by apimSubscription, bin(timestamp, 1m), agentID "
    "| order by timestamp asc"
)
print("Running the following Kusto query against App Insights:")
print(query)
output = utils.run(f'az monitor app-insights query --app {app_insights_name} -g {resource_group_name} --analytics-query "{query}"',
    f"App Insights query succeeded", f"App Insights query failed")

if output.success and output.json_data:
    table = output.json_data['tables'][0]
    df = pd.DataFrame(table.get("rows"), columns = [col.get("name") for col in table.get('columns')])
    df['timestamp'] = pd.to_datetime(df['timestamp']).dt.strftime('%H:%M')
    df.head()

Running the following Kusto query against App Insights:
 + "customMetrics " "| where name == 'Total Tokens' " "| where timestamp >= ago(1h) " "| extend parsedCustomDimensions = parse_json(customDimensions) " "| extend apimSubscription = tostring(parsedCustomDimensions.['Subscription ID']) " "| extend agentID = tostring(parsedCustomDimensions.['Agent ID']) " "| summarize TotalValue = sum(value) by apimSubscription, bin(timestamp, 1m), agentID " "| order by timestamp asc" + 
⚙️ Running: az monitor app-insights query --app insights-g3mjvytixvcoc -g lab-multi-model-failover --analytics-query  + "customMetrics " "| where name == 'Total Tokens' " "| where timestamp >= ago(1h) " "| extend parsedCustomDimensions = parse_json(customDimensions) " "| extend apimSubscription = tostring(parsedCustomDimensions.['Subscription ID']) " "| extend agentID = tostring(parsedCustomDimensions.['Agent ID']) " "| summarize TotalValue = sum(value) by apimSubscription, bin(timestamp, 1m), agentID " "| order by t

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
mpl.rcParams['figure.figsize'] = [15, 7]
if df.empty:
    print("No data to plot")
else:
    df_pivot = df.pivot(index='timestamp', columns='apimSubscription', values='TotalValue')
    ax = df_pivot.plot(kind='bar', stacked=True)
    plt.title('Total token usage over time by APIM Subscription')
    plt.xlabel('Time')
    plt.ylabel('Tokens')
    plt.legend(title='APIM Subscription')
    plt.show()

<a id='costs'></a>
### 📊 Analyze costs by subscription and agent

Query LAW (Azure Monitor Logs) to see token consumption across different subscriptions and agent executions. This data feeds into the FinOps framework for cost tracking.

In [20]:
log_analytics_workspace_id = "f8839ea4-23f3-41cf-a7c3-c8512d2463e6"

cost_query = (
    "let llmHeaderLogs = ApiManagementGatewayLlmLog "
    "| where DeploymentName != ''; "
    "let llmLogsWithSubscriptionId = llmHeaderLogs "
    "| join kind=leftouter ApiManagementGatewayLogs on CorrelationId "
    "| project "
    "    TimeGenerated, SubscriptionName = ApimSubscriptionId, DeploymentName, PromptTokens, CompletionTokens, TotalTokens; "
    "llmLogsWithSubscriptionId "
    "| join kind=inner ( "
    "    PRICING_CL "
    "    | summarize arg_max(TimeGenerated, *) by Model "
    "    | project Model, InputTokensPrice = coalesce(InputTokensPrice, 0.0), OutputTokensPrice = coalesce(OutputTokensPrice, 0.0) "
    "    ) "
    "    on $left.DeploymentName == $right.Model "
    "| extend InputCost = PromptTokens * InputTokensPrice "
    "| extend OutputCost = CompletionTokens * OutputTokensPrice "
    "| summarize "
    "    InputCost = sum(InputCost), OutputCost = sum(OutputCost) "
    "    by SubscriptionName, bin(TimeGenerated, 1m) "
    "| extend TotalCost = (InputCost + OutputCost) / 1000 "
    "| project TimeGenerated, SubscriptionName, TotalCost"
)

print("\n📊 Running cost analysis query against Log Analytics...")
import subprocess
rest_cmd = (
    f'az rest --method post '
    f'--url "https://api.loganalytics.io/v1/workspaces/{log_analytics_workspace_id}/query" '
    f'--headers "Content-Type=application/json" '
    f'--resource "https://api.loganalytics.io" '
    f'--body @-'
)

query_body = json.dumps({"query": cost_query, "timespan": "P30D"})
result = subprocess.run(rest_cmd, shell=True, input=query_body, capture_output=True, text=True)

if result.returncode == 0:
    response_data = json.loads(result.stdout)
    table = response_data['tables'][0]
    cols = [col.get("name") for col in table.get('columns')]
    rows = table.get("rows")
    df_cost = pd.DataFrame(rows, columns=cols)
    if df_cost.empty:
        print("No cost data available yet. Ensure pricing data has been loaded and requests have been made.")
    else:
        df_cost['TimeGenerated'] = pd.to_datetime(df_cost['TimeGenerated']).dt.strftime('%Y-%m-%d %H:%M')
        print(f"\n{'Time':<20} {'Subscription':<30} {'Total Cost ($)':<15}")
        print(f"{'-'*20} {'-'*30} {'-'*15}")
        for _, row in df_cost.iterrows():
            print(f"{row['TimeGenerated']:<20} {row['SubscriptionName']:<30} ${row['TotalCost']:<14.6f}")
        print(f"\n--- Summary ---")
        summary = df_cost.groupby('SubscriptionName')['TotalCost'].sum().reset_index()
        for _, row in summary.iterrows():
            print(f"  {row['SubscriptionName']:<30} Total: ${row['TotalCost']:.6f}")
    utils.print_ok("Cost analysis query succeeded")
else:
    utils.print_error(f"Cost analysis query failed: {result.stderr}")


📊 Running cost analysis query against Log Analytics...

Time                 Subscription                   Total Cost ($) 
-------------------- ------------------------------ ---------------
2026-03-25 11:37     subscription-finance           $3.352700      
2026-03-25 11:37     subscription-marketing         $1.790800      
2026-03-25 11:37     subscription-hr                $1.469100      
2026-03-25 11:38     subscription-finance           $0.011600      
2026-03-25 11:38                                    $0.000000      
2026-03-25 11:39     subscription-hr                $1.536400      
2026-03-25 11:39     subscription-marketing         $2.433900      
2026-03-25 11:39     subscription-finance           $4.589300      
2026-03-25 11:40     subscription-finance           $1.340200      
2026-03-25 11:40     subscription-marketing         $0.435100      
2026-03-25 11:40     subscription-hr                $0.778000      
2026-03-25 10:15     subscription-finance           $0.0020

<a id='clean'></a>
### 🗑️ Clean up resources

When you're finished with the lab, you should remove all your deployed resources from Azure to avoid extra charges and keep your Azure subscription uncluttered.
Use the [clean-up-resources notebook](clean-up-resources.ipynb) for that.